# Notebook 5: Training and optimization
We will re-use the code written in the previous notebooks for how to build a neural network to predict labels for the MNIST dataset images, and train said model.

Moreover, we will add a loop to evaluate the model's performance on a test dataset.

Note that, usually, we would split the dataset in train, validation and test subset, and only evaluate on the test one once the training is completely done.

### Load dataset and define the model

In [1]:
# Re-using the code from previous notebooks
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

training_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

In [2]:
# Network hyperparameters
learning_rate = 1e-3
batch_size = 64
epochs = 10

In [3]:
# Loss function: a common choice for classification is Cross Entropy
loss_fn = nn.CrossEntropyLoss()

In [4]:
# Initialize an optimizer. We'll use Stochastic Gradient Descent
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

### Define how to train and test the model
It is good practice to set the model to training mode when we train, end evaluation when we infer, as it is important for batch normalization and dropout layers (we don't have any here, but let's follow good practice anyways)

In [5]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)
        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        # Add some output
        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [7]:
def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0
    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Performance: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

### Train the model
Finally, let's use the optimization and test loop we defined to train our model, and save it once we're happy with the result

In [8]:
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.293531  [   64/60000]
loss: 2.307686  [ 6464/60000]
loss: 2.314296  [12864/60000]
loss: 2.285448  [19264/60000]
loss: 2.291675  [25664/60000]
loss: 2.294893  [32064/60000]
loss: 2.285482  [38464/60000]
loss: 2.282701  [44864/60000]
loss: 2.278952  [51264/60000]
loss: 2.280824  [57664/60000]
Test Performance: 
 Accuracy: 20.9%, Avg loss: 2.272939 

Epoch 2
-------------------------------
loss: 2.266232  [   64/60000]
loss: 2.277749  [ 6464/60000]
loss: 2.285873  [12864/60000]
loss: 2.251703  [19264/60000]
loss: 2.261798  [25664/60000]
loss: 2.262392  [32064/60000]
loss: 2.245903  [38464/60000]
loss: 2.254211  [44864/60000]
loss: 2.238832  [51264/60000]
loss: 2.241041  [57664/60000]
Test Performance: 
 Accuracy: 41.7%, Avg loss: 2.231520 

Epoch 3
-------------------------------
loss: 2.225493  [   64/60000]
loss: 2.232928  [ 6464/60000]
loss: 2.245997  [12864/60000]
loss: 2.200968  [19264/60000]
loss: 2.216034  [25664/60000]
loss: 2.213470

In [11]:
# We will save and load the model again to show how to do it
torch.save(model, 'model.pth')
model = torch.load('model.pth', weights_only = False) # Load with weights only = False only when you trust the source the model comes from